**Level 1**: Highest Density, Intensive Social Functions

• Universities and University Campuses (especially world-leading research institutions)

• High-Tech Businesses and Innovation Clusters

• Performance Centres, Art Galleries, and Museums

• Major Event Developments (e.g., Olympic Games facilities)

• Theme Parks and Large Leisure/Tourism Attractions (e.g., Euro Disneyland)

• Municipal and Provincial Government Offices

• Hospitals and Medical Centres (especially teaching/research facilities)

• High-Density, Mixed-Use Nodes (attracting employment, investment, and residential growth)

**Level 2**: Medium-High Density, Diversifying Social Functions

• Universities and University Campuses (regional)

• Hospitals and Health Services

• Cultural Facilities and Heritage Sites (leveraged for tourism)

• Government Administrative Offices

• Retail, Entertainment, and Leisure Precincts (including large retail tenants)

• Industrial Precincts and Manufacturing Facilities

• Business Headquarters and Financial Institutions

• Residential Areas for Commuters (in satellite cities)

• Port Facilities (connected by rail)

**Level 3:** Lower Density, Amenity/Lifestyle-Driven Social Functions

• Residential Areas (for retirees, lifestyle seekers, teleworkers, particularly in smaller settlements and villages well connected to a regional centre)

API: Overpass API
- is a Python library that provides an interface to the Overpass API. The Overpass API is a read-only API that allows users to query and retrieve custom-selected parts of OpenStreetMap (OSM) map data. 

In [3]:
!pip install overpy

  Using cached overpy-0.7-py3-none-any.whl.metadata (3.5 kB)
Using cached overpy-0.7-py3-none-any.whl (14 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
!pip install geopy

  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import overpy
import pandas as pd
from geopy.distance import geodesic

stations = [
    {"station": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"station": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"station": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"station": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"station": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"station": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"station": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"station": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"station": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"station": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"station": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"station": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"station": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"station": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

api = overpy.Overpass()

LEVEL_MAPPING = {
    # Level 1
    "university": "Level 1", "hospital": "Level 1", "arts_centre": "Level 1",
    "theatre": "Level 1", "cinema": "Level 1", "museum": "Level 1",
    "stadium": "Level 1", "town_hall": "Level 1", "courthouse": "Level 1",
    "embassy": "Level 1", "public_building": "Level 1",
    "theme_park": "Level 1", "zoo": "Level 1", "mall": "Level 1",
    "department_store": "Level 1",

    # Level 2
    "college": "Level 2", "school": "Level 2", "clinic": "Level 2",
    "doctors": "Level 2", "dentist": "Level 2", "monument": "Level 2",
    "memorial": "Level 2", "supermarket": "Level 2", "restaurant": "Level 2",
    "fast_food": "Level 2", "cafe": "Level 2", "pub": "Level 2",
    "bar": "Level 2", "industrial": "Level 2", "commercial": "Level 2",
    "bank": "Level 2", "atm": "Level 2", "bus_station": "Level 2",
    "railway_station": "Level 2",
    "retail": "Level 2", "office": "Level 2", "building_retail": "Level 2",
    "building_commercial": "Level 2", "building_office": "Level 2",

    # Level 3
    "residential": "Level 3", "nursing_home": "Level 3", "park": "Level 3",
    "playground": "Level 3", "dog_park": "Level 3", "picnic_site": "Level 3",
    "viewpoint": "Level 3", "hotel": "Level 3", "guesthouse": "Level 3",
    "bed_and_breakfast": "Level 3", "camp_site": "Level 3",
    "caravan_site": "Level 3", "convenience": "Level 3", "bakery": "Level 3",
    "florist": "Level 3", "butcher": "Level 3", "hairdresser": "Level 3",
    "garden": "Level 3", "parking": "Level 3", "parking_space": "Level 3",
    "parking_entrance": "Level 3", "house": "Level 3", "apartments": "Level 3"
}

def classify(tags):
    """Classify POI tags into Level 1/2/3 with building logic"""
    for k, v in tags.items():
        if v in LEVEL_MAPPING:
            return LEVEL_MAPPING[v]
    if "building" in tags and "name" in tags:
        return "Level 2"
    if "building" in tags:
        return "Level 3"
    return "Level 3"


lats = [st["lat"] for st in stations]
lons = [st["lon"] for st in stations]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)

query = f"""
(
    node({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    node({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    node({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    node({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    node({min_lat},{min_lon},{max_lat},{max_lon})[office];
    node({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    node({min_lat},{min_lon},{max_lat},{max_lon})[building];
    way({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    way({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    way({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    way({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    way({min_lat},{min_lon},{max_lat},{max_lon})[office];
    way({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    way({min_lat},{min_lon},{max_lat},{max_lon})[building];
);
out center;
"""
result = api.query(query)

records = []

for st in stations:
    st_coord = (st["lat"], st["lon"])
    for elem in (result.nodes + result.ways):
        if hasattr(elem, "lat"):
            latlon = (elem.lat, elem.lon)
        else:
            latlon = (elem.center_lat, elem.center_lon)
        dist = geodesic(st_coord, latlon).meters
        if dist <= 100:
            name = elem.tags.get("name", "Unnamed")
            tags = elem.tags
            category = next(iter(tags.values()), "unknown")
            level = classify(tags)
            records.append({
                "station": st["station"],
                "poi_name": name,
                "category": category,
                "lat": latlon[0],
                "lon": latlon[1],
                "distance_m": round(dist, 1),
                "level": level,
                "tags": dict(tags)
            })

df_100m = pd.DataFrame(records)
# df.to_csv("social_func.csv", index=False)

In [3]:
df.head(10)

,station,poi_name,category,lat,lon,distance_m,level,tags
0,Gungahlin Place,Coles,46,-35.1862687,149.1359771,83.2,Level 2,"{'addr:housenumber': '46', 'addr:street': 'Hib..."
1,Gungahlin Place,Unnamed,toilets,-35.1859503,149.1360954,65.8,Level 3,"{'amenity': 'toilets', 'fee': 'no'}"
2,Gungahlin Place,DaMingle,cafe,-35.1858324,149.1357939,35.7,Level 2,"{'amenity': 'cafe', 'name': 'DaMingle', 'smoki..."
3,Gungahlin Place,Unnamed,drinking_water,-35.1859790,149.1349820,59.1,Level 3,"{'amenity': 'drinking_water', 'check_date': '2..."
4,Gungahlin Place,Unnamed,bicycle_parking,-35.1859221,149.1349827,55.2,Level 3,"{'amenity': 'bicycle_parking', 'bicycle_parkin..."
5,Gungahlin Place,Unnamed,picnic_table,-35.1862600,149.1348458,90.0,Level 3,{'leisure': 'picnic_table'}
6,Gungahlin Place,Optus,Optus,-35.1857936,149.1355687,18.9,Level 3,"{'brand': 'Optus', 'brand:wikidata': 'Q865038'..."
7,Gungahlin Place,Unnamed,bench,-35.1860773,149.1346278,91.7,Level 3,"{'amenity': 'bench', 'backrest': 'yes'}"
8,Gungahlin Place,Unnamed,bench,-35.1860422,149.1346378,88.9,Level 3,"{'amenity': 'bench', 'backrest': 'yes'}"
9,Gungahlin Place,Unnamed,bench,-35.1861297,149.1349461,73.1,Level 3,"{'amenity': 'bench', 'backrest': 'yes'}"


In [6]:
import pandas as pd

df = pd.read_csv("social_func.csv")

level_counts = df.groupby(["station", "level"]).size().unstack(fill_value=0)

level_counts["soc_func"] = level_counts.idxmax(axis=1)

level_counts = level_counts.reset_index()
level_counts.to_csv("social_func_v2.csv", index=False)

print(level_counts)

level              station  Level 2  Level 3 soc_func
0            Alinga Street        1        1  Level 2
1      Dickson Interchange        1       21  Level 3
2        EPIC & Racecourse        0        2  Level 3
3           Elouera Street        1        4  Level 3
4          Gungahlin Place       10       16  Level 3
5             Ipima Street        0        7  Level 3
6         Macarthur Avenue        0        8  Level 3
7      Manning Clark North        2       21  Level 3
8          Mapleton Avenue        0        9  Level 3
9         Nullarbor Avenue        3       12  Level 3
10          Phillip Avenue        0        3  Level 3
11         Sandford Street        1        5  Level 3
12          Swinden Street        1       14  Level 3
13      Well Station Drive        0        7  Level 3


In [8]:
import overpy
import pandas as pd
from geopy.distance import geodesic

stations = [
    {"station": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"station": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"station": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"station": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"station": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"station": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"station": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"station": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"station": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"station": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"station": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"station": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"station": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"station": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

api = overpy.Overpass()

LEVEL_MAPPING = {
    # Level 1
    "university": "Level 1", "hospital": "Level 1", "arts_centre": "Level 1",
    "theatre": "Level 1", "cinema": "Level 1", "museum": "Level 1",
    "stadium": "Level 1", "town_hall": "Level 1", "courthouse": "Level 1",
    "embassy": "Level 1", "public_building": "Level 1",
    "theme_park": "Level 1", "zoo": "Level 1", "mall": "Level 1",
    "department_store": "Level 1",

    # Level 2
    "college": "Level 2", "school": "Level 2", "clinic": "Level 2",
    "doctors": "Level 2", "dentist": "Level 2", "monument": "Level 2",
    "memorial": "Level 2", "supermarket": "Level 2", "restaurant": "Level 2",
    "fast_food": "Level 2", "cafe": "Level 2", "pub": "Level 2",
    "bar": "Level 2", "industrial": "Level 2", "commercial": "Level 2",
    "bank": "Level 2", "atm": "Level 2", "bus_station": "Level 2",
    "railway_station": "Level 2",
    "retail": "Level 2", "office": "Level 2", "building_retail": "Level 2",
    "building_commercial": "Level 2", "building_office": "Level 2",

    # Level 3
    "residential": "Level 3", "nursing_home": "Level 3", "park": "Level 3",
    "playground": "Level 3", "dog_park": "Level 3", "picnic_site": "Level 3",
    "viewpoint": "Level 3", "hotel": "Level 3", "guesthouse": "Level 3",
    "bed_and_breakfast": "Level 3", "camp_site": "Level 3",
    "caravan_site": "Level 3", "convenience": "Level 3", "bakery": "Level 3",
    "florist": "Level 3", "butcher": "Level 3", "hairdresser": "Level 3",
    "garden": "Level 3", "parking": "Level 3", "parking_space": "Level 3",
    "parking_entrance": "Level 3", "house": "Level 3", "apartments": "Level 3"
}

def classify(tags):
    """Classify POI tags into Level 1/2/3 with building logic"""
    for k, v in tags.items():
        if v in LEVEL_MAPPING:
            return LEVEL_MAPPING[v]
    if "building" in tags and "name" in tags:
        return "Level 2"
    if "building" in tags:
        return "Level 3"
    return "Level 3"


lats = [st["lat"] for st in stations]
lons = [st["lon"] for st in stations]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)

query = f"""
(
    node({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    node({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    node({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    node({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    node({min_lat},{min_lon},{max_lat},{max_lon})[office];
    node({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    node({min_lat},{min_lon},{max_lat},{max_lon})[building];
    way({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    way({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    way({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    way({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    way({min_lat},{min_lon},{max_lat},{max_lon})[office];
    way({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    way({min_lat},{min_lon},{max_lat},{max_lon})[building];
);
out center;
"""
result = api.query(query)

records = []

for st in stations:
    st_coord = (st["lat"], st["lon"])
    for elem in (result.nodes + result.ways):
        if hasattr(elem, "lat"):
            latlon = (elem.lat, elem.lon)
        else:
            latlon = (elem.center_lat, elem.center_lon)
        dist = geodesic(st_coord, latlon).meters
        if dist <= 150:
            name = elem.tags.get("name", "Unnamed")
            tags = elem.tags
            category = next(iter(tags.values()), "unknown")
            level = classify(tags)
            records.append({
                "station": st["station"],
                "poi_name": name,
                "category": category,
                "lat": latlon[0],
                "lon": latlon[1],
                "distance_m": round(dist, 1),
                "level": level,
                "tags": dict(tags)
            })

df_150m = pd.DataFrame(records)
# df.to_csv("social_func.csv", index=False)

In [2]:
df_150m.head()

,station,poi_name,category,lat,lon,distance_m,level,tags
0,Gungahlin Place,Coles,46,-35.1862687,149.1359771,83.2,Level 2,"{'addr:housenumber': '46', 'addr:street': 'Hib..."
1,Gungahlin Place,Unnamed,post_box,-35.1861149,149.1370000,148.1,Level 3,"{'amenity': 'post_box', 'collection_times': 'M..."
2,Gungahlin Place,Unnamed,parking_entrance,-35.1863373,149.1365993,128.0,Level 3,"{'amenity': 'parking_entrance', 'maxheight': '..."
3,Gungahlin Place,Unnamed,toilets,-35.1859503,149.1360954,65.8,Level 3,"{'amenity': 'toilets', 'fee': 'no'}"
4,Gungahlin Place,Mycar,mycar Tyre & Auto,-35.1866646,149.1356206,114.5,Level 3,"{'brand': 'mycar Tyre & Auto', 'brand:wikidata..."


In [3]:
level_counts = df_150m.groupby(["station", "level"]).size().unstack(fill_value=0)
level_counts["soc_func"] = level_counts.idxmax(axis=1)
level_counts = level_counts.reset_index()
level_counts.to_csv("social_func_v2.csv", index=False)

print(level_counts)

level              station  Level 1  Level 2  Level 3 soc_func
0            Alinga Street        0        5       17  Level 3
1      Dickson Interchange        0        5       47  Level 3
2        EPIC & Racecourse        0        0        6  Level 3
3           Elouera Street        1        3       20  Level 3
4          Gungahlin Place        0       15       35  Level 3
5             Ipima Street        0        3       17  Level 3
6         Macarthur Avenue        0        2       31  Level 3
7      Manning Clark North        0        2       43  Level 3
8          Mapleton Avenue        0        0       21  Level 3
9         Nullarbor Avenue        0        6       24  Level 3
10          Phillip Avenue        0        1        8  Level 3
11         Sandford Street        0        4       10  Level 3
12          Swinden Street        0        1       25  Level 3
13      Well Station Drive        0        0       10  Level 3


In [11]:
import overpy
import pandas as pd
from geopy.distance import geodesic

stations = [
    {"station": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"station": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"station": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"station": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"station": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"station": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"station": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"station": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"station": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"station": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"station": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"station": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"station": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"station": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

api = overpy.Overpass()

LEVEL_MAPPING = {
    # Level 1
    "university": "Level 1", "hospital": "Level 1", "arts_centre": "Level 1",
    "theatre": "Level 1", "cinema": "Level 1", "museum": "Level 1",
    "stadium": "Level 1", "town_hall": "Level 1", "courthouse": "Level 1",
    "embassy": "Level 1", "public_building": "Level 1",
    "theme_park": "Level 1", "zoo": "Level 1", "mall": "Level 1",
    "department_store": "Level 1",

    # Level 2
    "college": "Level 2", "school": "Level 2", "clinic": "Level 2",
    "doctors": "Level 2", "dentist": "Level 2", "monument": "Level 2",
    "memorial": "Level 2", "supermarket": "Level 2", "restaurant": "Level 2",
    "fast_food": "Level 2", "cafe": "Level 2", "pub": "Level 2",
    "bar": "Level 2", "industrial": "Level 2", "commercial": "Level 2",
    "bank": "Level 2", "atm": "Level 2", "bus_station": "Level 2",
    "railway_station": "Level 2",
    "retail": "Level 2", "office": "Level 2", "building_retail": "Level 2",
    "building_commercial": "Level 2", "building_office": "Level 2",

    # Level 3
    "residential": "Level 3", "nursing_home": "Level 3", "park": "Level 3",
    "playground": "Level 3", "dog_park": "Level 3", "picnic_site": "Level 3",
    "viewpoint": "Level 3", "hotel": "Level 3", "guesthouse": "Level 3",
    "bed_and_breakfast": "Level 3", "camp_site": "Level 3",
    "caravan_site": "Level 3", "convenience": "Level 3", "bakery": "Level 3",
    "florist": "Level 3", "butcher": "Level 3", "hairdresser": "Level 3",
    "garden": "Level 3", "parking": "Level 3", "parking_space": "Level 3",
    "parking_entrance": "Level 3", "house": "Level 3", "apartments": "Level 3"
}

def classify(tags):
    """Classify POI tags into Level 1/2/3 with building logic"""
    for k, v in tags.items():
        if v in LEVEL_MAPPING:
            return LEVEL_MAPPING[v]
    if "building" in tags and "name" in tags:
        return "Level 2"
    if "building" in tags:
        return "Level 3"
    return "Level 3"


lats = [st["lat"] for st in stations]
lons = [st["lon"] for st in stations]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)

query = f"""
(
    node({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    node({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    node({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    node({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    node({min_lat},{min_lon},{max_lat},{max_lon})[office];
    node({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    node({min_lat},{min_lon},{max_lat},{max_lon})[building];
    way({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    way({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    way({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    way({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    way({min_lat},{min_lon},{max_lat},{max_lon})[office];
    way({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    way({min_lat},{min_lon},{max_lat},{max_lon})[building];
);
out center;
"""
result = api.query(query)

records = []

for st in stations:
    st_coord = (st["lat"], st["lon"])
    for elem in (result.nodes + result.ways):
        if hasattr(elem, "lat"):
            latlon = (elem.lat, elem.lon)
        else:
            latlon = (elem.center_lat, elem.center_lon)
        dist = geodesic(st_coord, latlon).meters
        if dist <= 200:
            name = elem.tags.get("name", "Unnamed")
            tags = elem.tags
            category = next(iter(tags.values()), "unknown")
            level = classify(tags)
            records.append({
                "station": st["station"],
                "poi_name": name,
                "category": category,
                "lat": latlon[0],
                "lon": latlon[1],
                "distance_m": round(dist, 1),
                "level": level,
                "tags": dict(tags)
            })

df_200m = pd.DataFrame(records)
# df.to_csv("social_func.csv", index=False)

In [5]:
df_200m.head()

,station,poi_name,category,lat,lon,distance_m,level,tags
0,Gungahlin Place,Coles,46,-35.1862687,149.1359771,83.2,Level 2,"{'addr:housenumber': '46', 'addr:street': 'Hib..."
1,Gungahlin Place,Big W,Hibberson Street,-35.1858740,149.1335016,182.2,Level 1,"{'addr:street': 'Hibberson Street', 'brand': '..."
2,Gungahlin Place,Unnamed,post_box,-35.1861149,149.1370000,148.1,Level 3,"{'amenity': 'post_box', 'collection_times': 'M..."
3,Gungahlin Place,Under Cover Big W,yes,-35.1860524,149.1335221,184.2,Level 3,"{'access': 'yes', 'amenity': 'parking', 'layer..."
4,Gungahlin Place,Unnamed,parking_entrance,-35.1863373,149.1365993,128.0,Level 3,"{'amenity': 'parking_entrance', 'maxheight': '..."


In [6]:
level_counts = df_200m.groupby(["station", "level"]).size().unstack(fill_value=0)
level_counts["soc_func"] = level_counts.idxmax(axis=1)
level_counts = level_counts.reset_index()
level_counts.to_csv("social_func_v2.csv", index=False)

print(level_counts)

level              station  Level 1  Level 2  Level 3 soc_func
0            Alinga Street        0        9       22  Level 3
1      Dickson Interchange        0       11       83  Level 3
2        EPIC & Racecourse        0        0       15  Level 3
3           Elouera Street        1        8       34  Level 3
4          Gungahlin Place        1       21       44  Level 3
5             Ipima Street        0        4       41  Level 3
6         Macarthur Avenue        0        6       50  Level 3
7      Manning Clark North        0        4       62  Level 3
8          Mapleton Avenue        0        0       52  Level 3
9         Nullarbor Avenue        0        8       37  Level 3
10          Phillip Avenue        0        1       11  Level 3
11         Sandford Street        0        5       16  Level 3
12          Swinden Street        0        2       37  Level 3
13      Well Station Drive        0        1       12  Level 3


In [12]:
import overpy
import pandas as pd
from geopy.distance import geodesic

stations = [
    {"station": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"station": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"station": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"station": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"station": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"station": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"station": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"station": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"station": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"station": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"station": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"station": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"station": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"station": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

api = overpy.Overpass()

LEVEL_MAPPING = {
    # Level 1
    "university": "Level 1", "hospital": "Level 1", "arts_centre": "Level 1",
    "theatre": "Level 1", "cinema": "Level 1", "museum": "Level 1",
    "stadium": "Level 1", "town_hall": "Level 1", "courthouse": "Level 1",
    "embassy": "Level 1", "public_building": "Level 1",
    "theme_park": "Level 1", "zoo": "Level 1", "mall": "Level 1",
    "department_store": "Level 1",

    # Level 2
    "college": "Level 2", "school": "Level 2", "clinic": "Level 2",
    "doctors": "Level 2", "dentist": "Level 2", "monument": "Level 2",
    "memorial": "Level 2", "supermarket": "Level 2", "restaurant": "Level 2",
    "fast_food": "Level 2", "cafe": "Level 2", "pub": "Level 2",
    "bar": "Level 2", "industrial": "Level 2", "commercial": "Level 2",
    "bank": "Level 2", "atm": "Level 2", "bus_station": "Level 2",
    "railway_station": "Level 2",
    "retail": "Level 2", "office": "Level 2", "building_retail": "Level 2",
    "building_commercial": "Level 2", "building_office": "Level 2",

    # Level 3
    "residential": "Level 3", "nursing_home": "Level 3", "park": "Level 3",
    "playground": "Level 3", "dog_park": "Level 3", "picnic_site": "Level 3",
    "viewpoint": "Level 3", "hotel": "Level 3", "guesthouse": "Level 3",
    "bed_and_breakfast": "Level 3", "camp_site": "Level 3",
    "caravan_site": "Level 3", "convenience": "Level 3", "bakery": "Level 3",
    "florist": "Level 3", "butcher": "Level 3", "hairdresser": "Level 3",
    "garden": "Level 3", "parking": "Level 3", "parking_space": "Level 3",
    "parking_entrance": "Level 3", "house": "Level 3", "apartments": "Level 3"
}

def classify(tags):
    """Classify POI tags into Level 1/2/3 with building logic"""
    for k, v in tags.items():
        if v in LEVEL_MAPPING:
            return LEVEL_MAPPING[v]
    if "building" in tags and "name" in tags:
        return "Level 2"
    if "building" in tags:
        return "Level 3"
    return "Level 3"


lats = [st["lat"] for st in stations]
lons = [st["lon"] for st in stations]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)

query = f"""
(
    node({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    node({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    node({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    node({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    node({min_lat},{min_lon},{max_lat},{max_lon})[office];
    node({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    node({min_lat},{min_lon},{max_lat},{max_lon})[building];
    way({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    way({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    way({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    way({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    way({min_lat},{min_lon},{max_lat},{max_lon})[office];
    way({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    way({min_lat},{min_lon},{max_lat},{max_lon})[building];
);
out center;
"""
result = api.query(query)

records = []

for st in stations:
    st_coord = (st["lat"], st["lon"])
    for elem in (result.nodes + result.ways):
        if hasattr(elem, "lat"):
            latlon = (elem.lat, elem.lon)
        else:
            latlon = (elem.center_lat, elem.center_lon)
        dist = geodesic(st_coord, latlon).meters
        if dist <= 250:
            name = elem.tags.get("name", "Unnamed")
            tags = elem.tags
            category = next(iter(tags.values()), "unknown")
            level = classify(tags)
            records.append({
                "station": st["station"],
                "poi_name": name,
                "category": category,
                "lat": latlon[0],
                "lon": latlon[1],
                "distance_m": round(dist, 1),
                "level": level,
                "tags": dict(tags)
            })

df_250m = pd.DataFrame(records)
# df.to_csv("social_func.csv", index=False)

In [8]:
df_250m.head()

,station,poi_name,category,lat,lon,distance_m,level,tags
0,Gungahlin Place,Coles,46,-35.1862687,149.1359771,83.2,Level 2,"{'addr:housenumber': '46', 'addr:street': 'Hib..."
1,Gungahlin Place,Big W,Hibberson Street,-35.1858740,149.1335016,182.2,Level 1,"{'addr:street': 'Hibberson Street', 'brand': '..."
2,Gungahlin Place,Unnamed,post_box,-35.1861149,149.1370000,148.1,Level 3,"{'amenity': 'post_box', 'collection_times': 'M..."
3,Gungahlin Place,Under Cover Big W,yes,-35.1860524,149.1335221,184.2,Level 3,"{'access': 'yes', 'amenity': 'parking', 'layer..."
4,Gungahlin Place,Unnamed,parking_entrance,-35.1863373,149.1365993,128.0,Level 3,"{'amenity': 'parking_entrance', 'maxheight': '..."


In [9]:
level_counts = df_250m.groupby(["station", "level"]).size().unstack(fill_value=0)
level_counts["soc_func"] = level_counts.idxmax(axis=1)
level_counts = level_counts.reset_index()
level_counts.to_csv("social_func_v2.csv", index=False)

print(level_counts)

level              station  Level 1  Level 2  Level 3 soc_func
0            Alinga Street        0       16       24  Level 3
1      Dickson Interchange        0       23      113  Level 3
2        EPIC & Racecourse        0        2       28  Level 3
3           Elouera Street        1       23       60  Level 3
4          Gungahlin Place        1       25       47  Level 3
5             Ipima Street        0        7       66  Level 3
6         Macarthur Avenue        0        7       86  Level 3
7      Manning Clark North        0        4       84  Level 3
8          Mapleton Avenue        0        0       70  Level 3
9         Nullarbor Avenue        0        9       52  Level 3
10          Phillip Avenue        0        1       14  Level 3
11         Sandford Street        0       10       23  Level 3
12          Swinden Street        0        3       51  Level 3
13      Well Station Drive        0        1       19  Level 3


In [13]:
import overpy
import pandas as pd
from geopy.distance import geodesic

stations = [
    {"station": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"station": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"station": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"station": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"station": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"station": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"station": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"station": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"station": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"station": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"station": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"station": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"station": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"station": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

api = overpy.Overpass()

LEVEL_MAPPING = {
    # Level 1
    "university": "Level 1", "hospital": "Level 1", "arts_centre": "Level 1",
    "theatre": "Level 1", "cinema": "Level 1", "museum": "Level 1",
    "stadium": "Level 1", "town_hall": "Level 1", "courthouse": "Level 1",
    "embassy": "Level 1", "public_building": "Level 1",
    "theme_park": "Level 1", "zoo": "Level 1", "mall": "Level 1",
    "department_store": "Level 1",

    # Level 2
    "college": "Level 2", "school": "Level 2", "clinic": "Level 2",
    "doctors": "Level 2", "dentist": "Level 2", "monument": "Level 2",
    "memorial": "Level 2", "supermarket": "Level 2", "restaurant": "Level 2",
    "fast_food": "Level 2", "cafe": "Level 2", "pub": "Level 2",
    "bar": "Level 2", "industrial": "Level 2", "commercial": "Level 2",
    "bank": "Level 2", "atm": "Level 2", "bus_station": "Level 2",
    "railway_station": "Level 2",
    "retail": "Level 2", "office": "Level 2", "building_retail": "Level 2",
    "building_commercial": "Level 2", "building_office": "Level 2",

    # Level 3
    "residential": "Level 3", "nursing_home": "Level 3", "park": "Level 3",
    "playground": "Level 3", "dog_park": "Level 3", "picnic_site": "Level 3",
    "viewpoint": "Level 3", "hotel": "Level 3", "guesthouse": "Level 3",
    "bed_and_breakfast": "Level 3", "camp_site": "Level 3",
    "caravan_site": "Level 3", "convenience": "Level 3", "bakery": "Level 3",
    "florist": "Level 3", "butcher": "Level 3", "hairdresser": "Level 3",
    "garden": "Level 3", "parking": "Level 3", "parking_space": "Level 3",
    "parking_entrance": "Level 3", "house": "Level 3", "apartments": "Level 3"
}

def classify(tags):
    """Classify POI tags into Level 1/2/3 with building logic"""
    for k, v in tags.items():
        if v in LEVEL_MAPPING:
            return LEVEL_MAPPING[v]
    if "building" in tags and "name" in tags:
        return "Level 2"
    if "building" in tags:
        return "Level 3"
    return "Level 3"


lats = [st["lat"] for st in stations]
lons = [st["lon"] for st in stations]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)

query = f"""
(
    node({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    node({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    node({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    node({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    node({min_lat},{min_lon},{max_lat},{max_lon})[office];
    node({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    node({min_lat},{min_lon},{max_lat},{max_lon})[building];
    way({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    way({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    way({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    way({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    way({min_lat},{min_lon},{max_lat},{max_lon})[office];
    way({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    way({min_lat},{min_lon},{max_lat},{max_lon})[building];
);
out center;
"""
result = api.query(query)

records = []

for st in stations:
    st_coord = (st["lat"], st["lon"])
    for elem in (result.nodes + result.ways):
        if hasattr(elem, "lat"):
            latlon = (elem.lat, elem.lon)
        else:
            latlon = (elem.center_lat, elem.center_lon)
        dist = geodesic(st_coord, latlon).meters
        if dist <= 300:
            name = elem.tags.get("name", "Unnamed")
            tags = elem.tags
            category = next(iter(tags.values()), "unknown")
            level = classify(tags)
            records.append({
                "station": st["station"],
                "poi_name": name,
                "category": category,
                "lat": latlon[0],
                "lon": latlon[1],
                "distance_m": round(dist, 1),
                "level": level,
                "tags": dict(tags)
            })

df_300m = pd.DataFrame(records)
# df.to_csv("social_func.csv", index=False)

In [11]:
level_counts = df_300m.groupby(["station", "level"]).size().unstack(fill_value=0)
level_counts["soc_func"] = level_counts.idxmax(axis=1)
level_counts = level_counts.reset_index()
level_counts.to_csv("social_func_v2.csv", index=False)

print(level_counts)

level              station  Level 1  Level 2  Level 3 soc_func
0            Alinga Street        0       22       30  Level 3
1      Dickson Interchange        0       54      163  Level 3
2        EPIC & Racecourse        0        3       34  Level 3
3           Elouera Street        1       46       91  Level 3
4          Gungahlin Place        1       29       54  Level 3
5             Ipima Street        0        8      102  Level 3
6         Macarthur Avenue        0        8      125  Level 3
7      Manning Clark North        0        5      107  Level 3
8          Mapleton Avenue        0        0       86  Level 3
9         Nullarbor Avenue        0        9       62  Level 3
10          Phillip Avenue        0        1       19  Level 3
11         Sandford Street        0       16       37  Level 3
12          Swinden Street        0        3       59  Level 3
13      Well Station Drive        0        4       32  Level 3


In [15]:
import overpy
import pandas as pd
from geopy.distance import geodesic

stations = [
    {"station": "Gungahlin Place", "lat": -35.185639, "lon": 149.135481},
    {"station": "Manning Clark North", "lat": -35.1869861, "lon": 149.1433722},
    {"station": "Mapleton Avenue", "lat": -35.193389, "lon": 149.150972},
    {"station": "Nullarbor Avenue", "lat": -35.200556, "lon": 149.149305},
    {"station": "Well Station Drive", "lat": -35.2091361, "lon": 149.1472889},
    {"station": "Sandford Street", "lat": -35.221703, "lon": 149.144722},
    {"station": "EPIC & Racecourse", "lat": -35.228438, "lon": 149.143558},
    {"station": "Phillip Avenue", "lat": -35.235895, "lon": 149.143757},
    {"station": "Swinden Street", "lat": -35.2445364, "lon": 149.1344672},
    {"station": "Dickson Interchange", "lat": -35.2505862, "lon": 149.1336686},
    {"station": "Macarthur Avenue", "lat": -35.259988 , "lon": 149.132039},
    {"station": "Ipima Street", "lat": -35.265918, "lon": 149.131244},
    {"station": "Elouera Street", "lat": -35.272761, "lon": 149.130168},
    {"station": "Alinga Street", "lat": -35.277930, "lon": 149.129330}
]

api = overpy.Overpass()

LEVEL_MAPPING = {
    # Level 1
    "university": "Level 1", "hospital": "Level 1", "arts_centre": "Level 1",
    "theatre": "Level 1", "cinema": "Level 1", "museum": "Level 1",
    "stadium": "Level 1", "townhall": "Level 1", "courthouse": "Level 1",
    "embassy": "Level 1", "public_building": "Level 1",
    "theme_park": "Level 1", "zoo": "Level 1", "mall": "Level 1",
    "department_store": "Level 1", "government": "Level 1",

    # Level 2
    "college": "Level 2", "school": "Level 2", "clinic": "Level 2",
    "doctors": "Level 2", "dentist": "Level 2", "monument": "Level 2",
    "memorial": "Level 2", "supermarket": "Level 2", "restaurant": "Level 2",
    "fast_food": "Level 2", "cafe": "Level 2", "pub": "Level 2",
    "bar": "Level 2", "industrial": "Level 2", "commercial": "Level 2",
    "bank": "Level 2", "atm": "Level 2", "bus_station": "Level 2",
    "railway_station": "Level 2", "retail": "Level 2", "office": "Level 2",
    "building_retail": "Level 2", "building_commercial": "Level 2",
    "building_office": "Level 2",

    # Level 3
    "residential": "Level 3", "nursing_home": "Level 3", "park": "Level 3",
    "playground": "Level 3", "dog_park": "Level 3", "picnic_site": "Level 3",
    "viewpoint": "Level 3", "hotel": "Level 3", "guesthouse": "Level 3",
    "bed_and_breakfast": "Level 3", "camp_site": "Level 3",
    "caravan_site": "Level 3", "convenience": "Level 3", "bakery": "Level 3",
    "florist": "Level 3", "butcher": "Level 3", "hairdresser": "Level 3",
    "garden": "Level 3", "parking": "Level 3", "house": "Level 3",
    "apartments": "Level 3"
}

def classify(tags):
    """Classify OSM element into Level 1/2/3."""
    for k, v in tags.items():
        if v in LEVEL_MAPPING:
            return LEVEL_MAPPING[v]
        if k in LEVEL_MAPPING:
            return LEVEL_MAPPING[k]
    if "building" in tags:
        return "Level 3"
    return "Level 3"

lats = [st["lat"] for st in stations]
lons = [st["lon"] for st in stations]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)

query = f"""
(
    node({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    node({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    node({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    node({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    node({min_lat},{min_lon},{max_lat},{max_lon})[office];
    node({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    node({min_lat},{min_lon},{max_lat},{max_lon})[building];

    way({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    way({min_lat},{min_lon},{max_lat},{max_lon})[shop];
    way({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    way({min_lat},{min_lon},{max_lat},{max_lon})[leisure];
    way({min_lat},{min_lon},{max_lat},{max_lon})[office];
    way({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
    way({min_lat},{min_lon},{max_lat},{max_lon})[building];

    relation({min_lat},{min_lon},{max_lat},{max_lon})[amenity];
    relation({min_lat},{min_lon},{max_lat},{max_lon})[tourism];
    relation({min_lat},{min_lon},{max_lat},{max_lon})[office];
    relation({min_lat},{min_lon},{max_lat},{max_lon})[landuse];
);
out center;
"""
result = api.query(query)

records = []

for st in stations:
    st_coord = (st["lat"], st["lon"])
    for elem in (result.nodes + result.ways + result.relations):
        if hasattr(elem, "lat"):  # node
            latlon = (elem.lat, elem.lon)
        else:  # way/relation (center)
            latlon = (elem.center_lat, elem.center_lon)
        dist = geodesic(st_coord, latlon).meters
        if dist <= 500:
            name = elem.tags.get("name", "Unnamed")
            level = classify(elem.tags)
            records.append({
                "station": st["station"],
                "poi_name": name,
                "tags": dict(elem.tags),
                "lat": latlon[0],
                "lon": latlon[1],
                "distance_m": round(dist, 1),
                "level": level
            })

df = pd.DataFrame(records)


In [13]:
df.head()

,station,poi_name,tags,lat,lon,distance_m,level
0,Gungahlin Place,Coles,"{'addr:housenumber': '46', 'addr:street': 'Hib...",-35.1862687,149.1359771,83.2,Level 2
1,Gungahlin Place,Big W,"{'addr:street': 'Hibberson Street', 'brand': '...",-35.1858740,149.1335016,182.2,Level 1
2,Gungahlin Place,Unnamed,"{'amenity': 'post_box', 'collection_times': 'M...",-35.1861149,149.1370000,148.1,Level 3
3,Gungahlin Place,Under Cover Big W,"{'access': 'yes', 'amenity': 'parking', 'layer...",-35.1860524,149.1335221,184.2,Level 3
4,Gungahlin Place,Unnamed,"{'amenity': 'parking_entrance', 'maxheight': '...",-35.1863373,149.1365993,128.0,Level 3


In [26]:
# Weighted Decision Matrix
level_counts = df.groupby(["station", "level"]).size().unstack(fill_value=0)
weights = {"Level 1": 100, "Level 2": 10, "Level 3": 1}

def weighted_decision_matrix(row):
    """
    Assigns dominant level to a station using a weighted decision matrix.
    """
    # Multiply each level count by its weight
    scores = {level: row[level] * weight for level, weight in weights.items()}
    
    # Identify the level with the highest weighted score
    dominant_level = max(scores, key=scores.get)
    return dominant_level

# Apply weighted decision matrix
level_counts["soc_func"] = level_counts.apply(weighted_decision_matrix, axis=1)

# Reset index so 'station' becomes a normal column
level_counts = level_counts.reset_index()
level_counts.columns.name = None 


In [28]:
level_counts.head(15)

,station,Level 1,Level 2,Level 3,soc_func
0,Alinga Street,2,72,363,Level 2
1,Dickson Interchange,1,104,327,Level 2
2,EPIC & Racecourse,5,2,66,Level 1
3,Elouera Street,2,94,307,Level 2
4,Gungahlin Place,3,34,114,Level 2
5,Ipima Street,0,10,336,Level 3
6,Macarthur Avenue,1,13,308,Level 3
7,Manning Clark North,0,7,187,Level 3
8,Mapleton Avenue,0,0,129,Level 3
9,Nullarbor Avenue,0,11,87,Level 2


In [29]:
level_counts.to_csv("social_function.csv", index=False)